# Spatial evolution — the code track

PHIL 2001, *Ethics and Evolutionary Games*.

The widget lets you *use* the model. This notebook lets you *change* it. Everything here
runs on `spatial.py`, which is a single readable file — if you want to know what the model
actually does, that is the thing to read, and this notebook is a guided tour of it.

**The one idea.** Players sit at fixed positions, play a game with their *neighbours*, and
revise by looking at how those neighbours did. Space does not act on strategies directly.
It works by producing **assortment** — making like meet like — and assortment is the lever.
This notebook builds up to measuring exactly that, so you can hand the number to the
replicator explorer and ask whether a well-mixed population with that much correlation
would do the same thing.

In [ ]:
# Setup. On Colab this downloads the two model files; locally it does nothing.
import os

if not os.path.exists("spatial.py"):
    BASE = "https://raw.githubusercontent.com/rorysmead/phil2001-spatial/main"
    for _f in ("spatial.py", "render.py"):
        os.system(f"curl -sO {BASE}/{_f}")

import matplotlib.pyplot as plt
import numpy as np

import render as R
import spatial as sp

print("rules:", ", ".join(sp.RULES))
print("games:", ", ".join(sp.GAMES))

## 1. A first run

Three objects and you have a model:

- a **payoff matrix** `A`, where `A[i, j]` is what strategy `i` earns against strategy `j`;
- a **`Model`**, which is the lattice, the neighbourhood and the update rule;
- a **`Sim`**, which is a `Model` plus a starting lattice and a **seed**.

The seed matters more than it looks. Every random choice in a run goes through it, so a run
is completely reproducible — "run it a few times and see if the result is reliable" only
means something if you can also say *which* runs you did.

In [ ]:
A, names = sp.GAMES["Prisoner's Dilemma - Nowak & May (b=1.85)"]

model = sp.Model(
    A=A,
    rows=100, cols=100,          # the lattice
    neighbourhood="Moore (8)",   # who you play
    wrap=True,                   # a torus: no edges
    self_play=True,              # Nowak & May have each cell play itself too
    rule="imitate_best",         # copy whoever did best nearby
    schedule="synchronous",      # everyone revises at once
)

sim = sp.Sim(model, seed=0).run(200)
R.dashboard(sim, names=names)
plt.show()

Defection strictly dominates — a defector does better than a cooperator against *any* single
opponent — and yet cooperators are still there, in clusters. That is the whole phenomenon.
Clusters of cooperators do well because they mostly meet each other.

## 2. What the result rests on

Nowak & May (*Nature* 359, 1992) report cooperators settling near **0.318**. Before treating
that as a fact about the Prisoner's Dilemma, find out what it is a fact *about*. Change one
assumption at a time and watch.

In [ ]:
import dataclasses

def final_C(model, seeds=(0, 1, 2), generations=200):
    """Cooperator frequency at the end, for several seeds."""
    return [round(float(sp.Sim(model, seed=s).run(generations).snapshot()["frequencies"][0]), 3)
            for s in seeds]

print("as published (100x100, self-play on) ", final_C(model))
print("without self-interaction             ", final_C(dataclasses.replace(model, self_play=False)))
print("on a small 40x40 lattice             ", final_C(dataclasses.replace(model, rows=40, cols=40)))

Two assumptions turn out to be carrying the result:

1. **Self-interaction.** Nowak & May have each player play its eight neighbours *and its own
   site* (they describe a 3×3 territory). That adds `A[i,i]` to everyone — and in this game
   `A[C,C] = 1` while `A[D,D] = 0`, so it is a flat bonus paid to cooperators only, worth
   about one extra neighbour. Switch it off and cooperation dies.
2. **Lattice size.** At 40×40 individual runs fixate at 0 or 1. The 0.318 coexistence needs
   roughly 100×100. A small lattice is not a fast version of a big one; it is a different
   model.

And a third, which is someone else's objection. Huberman & Glance (*PNAS* 90, 1993) argued
that the spatial chaos depends on every cell updating in lockstep — a shared global clock is
a strong thing to hang a result on. Test it:

In [ ]:
small = dataclasses.replace(model, rows=50, cols=50)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, sched in zip(axes, ("synchronous", "sequential")):
    s = sp.Sim(dataclasses.replace(small, schedule=sched), seed=0).run(60)
    R.draw_lattice(s, names=names, ax=ax, title=f"{sched}\nC = {s.snapshot()['frequencies'][0]:.2f}")
plt.show()

## 3. Assortment: the number that connects the two tools

`sp.assortment` takes every (site, neighbour) pair and asks how often the two play the same
strategy, then compares that with what you would get by reshuffling the very same strategies
at random over the lattice:

$$r = \frac{P_{\text{same}} - P_{\text{random}}}{1 - P_{\text{random}}}$$

`r = 0` means the lattice is doing nothing. `r > 0` is positive assortment — the altruism
lever, the same `r` as the replicator explorer's slider. `r < 0` is anti-assortment, the
mirror lever behind spite.

In [ ]:
off = sp.neighbour_offsets("vn", 1)
N = 60   # r is a sample statistic: on a small lattice chance alone moves it a few per cent
checker = np.indices((N, N)).sum(axis=0) % 2                 # perfect anti-assortment
blocks = np.zeros((N, N), np.int64); blocks[:, N // 2:] = 1  # perfect segregation
scatter = np.random.default_rng(0).integers(0, 2, (N, N))

for label, g in (("checkerboard", checker), ("random scatter", scatter), ("two blocks", blocks)):
    print(f"{label:>16}:  r = {sp.assortment(g, off, wrap=True, k=2):+.3f}")

In [ ]:
# Watch a lattice BUILD its own assortment from a random start.
sim = sp.Sim(model, seed=0).run(200)
r = sim.series("assortment")
print(f"r at generation 0:   {r[0]:+.3f}")
print(f"r at generation 200: {r[-1]:+.3f}")
print("\nNow open the replicator explorer, enter this same payoff matrix,")
print(f"and set its assortment slider to r = {r[-1]:.2f}. Does the well-mixed")
print("model predict the cooperator frequency the lattice actually reached?")
print(f"(the lattice reached {sim.snapshot()['frequencies'][0]:.3f})")

That comparison — not the pictures — is the result. If the well-mixed model with that much
correlation agrees, then space mattered *only* through assortment, and you have explained the
lattice rather than just watched it. If it disagrees, the gap is the interesting part: the
lattice is doing something a single correlation parameter cannot capture.

## 4. Where selection acts

The two birth–death rules are a controlled experiment on a modelling choice that usually goes
unexamined. Same births, same deaths, same lattice, same game — only the step that selection
acts on moves.

- **`death_birth`** — a site dies at random; its neighbours compete to fill it in proportion
  to payoff. Selection on **birth**. (This is the death–birth updating of evolutionary graph
  theory; Ohtsuki et al., *Nature* 441, 2006, give the condition b/c > k under weak selection.)
- **`death_selection`** — doing badly is what kills you; the gap is then filled by a *random*
  neighbour. Selection on **death**.

`selection` (w) sets how much payoffs matter at all: `w = 1` is full strength, `w = 0` is pure
drift. It is not a detail — the b/c > k result is a weak-selection result and does not survive
being run at w = 1.

In [ ]:
print("Donation game, 60x60, Moore-8 (so k = 8). Final cooperator frequency.\n")
print(f"{'b/c':>5} | {'death_birth w=1':>16} | {'death_birth w=0.02':>19} | {'death_selection w=0.02':>23}")
for bc in (2, 8, 20):
    row = []
    for rule, w in (("death_birth", 1.0), ("death_birth", 0.02), ("death_selection", 0.02)):
        m = sp.Model(A=sp.prisoners_dilemma(b=bc, c=1.0), rows=60, cols=60,
                     rule=rule, selection=w, death_rate=1.0)
        row.append(np.mean(final_C(m, seeds=(0, 1), generations=300)))
    print(f"{bc:>5} | {row[0]:>16.3f} | {row[1]:>19.3f} | {row[2]:>23.3f}")

## 5. Build your own starting lattice

The starting configuration is not a detail either. A random 50/50 scatter and a single compact
cluster of the same size are different experiments: the cluster already *has* the assortment
the scatter has to build. Pass any array you like as `grid=`.

In [ ]:
# The classic: one small cluster of cooperators in a sea of defectors. Can it grow?
g = np.ones((60, 60), np.int64)          # 1 = Defect everywhere
g[28:32, 28:32] = 0                      # a 4x4 block of Cooperators

sim = sp.Sim(dataclasses.replace(model, rows=60, cols=60), grid=g, seed=0).run(80)
R.dashboard(sim, names=names)
plt.show()
print("final cooperator frequency:", sim.snapshot()["frequencies"][0])

## 6. Exercises

1. **Find the boundary.** For the Nowak & May game, at which value of `b` do cooperators stop
   surviving? Sweep `b` and plot final cooperator frequency against it. Use several seeds per
   value of `b` — you have already seen how much runs vary.
2. **Does the neighbourhood matter?** Repeat a result you like under von Neumann (4), Moore
   (8) and Moore r=2 (24). If it survives all three it is about the game; if it does not, it
   is about the neighbourhood.
3. **Spite.** Run `sp.GAMES["Spite (cost c=1, harm h=3)"]`, where every payoff is negative.
   Does the lattice help harming behaviour or hinder it? What does `r` do? (Positive
   assortment supports altruism; spite is supposed to be the mirror, needing *negative*
   assortment. Does a lattice ever produce that?)
4. **Break something.** Write your own update rule — a function with the same signature as
   those in `spatial.py` section 3 — add it to `sp.RULES`, and see whether a result you
   believed survives it.
5. **Check the tool.** Run `python3 tests.py`. Every claim in this notebook that could be
   checked mechanically is checked there. Add a test for something you think might be wrong.